# DigIA
## Pipeline de Visão Computacional e Testes de Generalização Extrema

In [ ]:
# carregamento das bibliotecas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.base import clone
import time
import glob
import os
import cv2
import numpy as np
from scipy import ndimage
import math
import random

#### Fase 1: Carregamento e Análise Exploratória de Imagens (EDA)

In [ ]:
# Carregando dataset MNIST a partir da OpenML
print("Baixando o dataset MNIST...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
print("Finalizou o processo de baixar o dataset MNIST!")

In [ ]:
# Separação entre features e target
X, y = mnist.data, mnist.target


In [ ]:
# Exploração da dimensionalidade das matrizes
print (f'========>> A matriz X possui {X.shape[0]} amostras, sendo {X.shape[1]} pixels por imagem')
print (f'========>> A matriz y é unidimensional e possui {y.shape[0]} rótulos / targets\n')
print (f'========>> Matriz de pixels de uma amostra:\n\n {X[10000]}')


In [ ]:
# Verificação da distribuição das classes e balanceamento do dataset
rotulos, freq_abs = np.unique(y, return_counts=True)
freq_perc = np.round(freq_abs / freq_abs.sum() * 100, decimals = 2)
print (f'Rótulos únicos: {rotulos}')
print (f'Frequências absolutas: {freq_abs}')
print (f'Frequências percentuais: {freq_perc}')

As classes presentes no dataset são 10 dígitos entre 0 e 9<br>
O dataset tem uma distribuição balanceada de um modo geral

In [ ]:
# Averiguação dos tipos de dados
print (f'Tipos de dados da matriz X: {X.dtype}')
print (f'Tipos de dados da matriz y: {y.dtype}')

A matriz de rótulos contém dados do tipo `object`, que no caso são strings. Vamos transformar estes dados para inteiros.

In [ ]:
# Alteração do tipo de dados da matriz de rótulos para inteiro
y = y.astype(np.uint8)

In [ ]:
# Geração de grade visual com 24 amostras aleatóras
random_idx = np.random.randint(0, X.shape[0], size=24)

fig, axes = plt.subplots(4, 6, figsize=[10, 7])
axes = axes.ravel()

for i, idx_i in enumerate(random_idx):
    xval = X[idx_i].reshape(28, 28)
    yval = int(y[idx_i])
    axes[i].imshow(xval, cmap='gray', vmin=0, vmax=255)
    axes[i].axis('off')
    axes[i].set_title(f"Dígito {yval}", fontsize=11, fontweight='bold')

plt.suptitle("24 Amostras Aleatórias do MNIST", fontsize=16)
plt.tight_layout()
plt.show()


Cada amostra do Dataset MNIST consiste de uma matriz unidimensional com tamanho 784 e são a representação achatada de uma figura com 28x28 pixels na qual cada um deles possui um valor que representa a intensidade de luz emitida por ele. Aqui, o canal de cor é único e portanto apenas um valor é o suficiente e pode variar de 0 (ausência de luz) a 255 (intensidade total de luz).<br>

Para representar as imagens na tela, é necessário rearranjar a matriz unidimensional e uma matriz bidimensional de dimensões 28x28. A função `reshape()`do Numpy é fundamental para isto.<br>

A plotagem utilizou uma representação baseada em tons de cinza (Grayscale), sendo que o fundo das figuras é preto e as regiões que contém os traços aproximam-se mais do branco. Comparando a figura com a matriz numérica observamos esta correspondência pois a maior parte dos pixels tem valor 0 (preto)

#### Fase 2: Pipeline de Pré-processamento e Divisão dos Dados

In [ ]:
# divisão estratificada dos dados em conjunto de treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify = y, random_state=42)

In [ ]:
# escalonamento dos dados
scaler = MinMaxScaler()

X_train_escalonado = scaler.fit_transform(X_train)
X_test_escalonado = scaler.transform(X_test)

Importância do escalonamento: o escalonador `MinMaxScaler` por padrão redimensiona os dados numéricos proporcionalmente para que resultem em valores entre 0 e 1. Modelos lineares e/ou que utilizam distâncias métricas podem ter oscilações muito grandes com valores com magnitudes diferentes e terão dificuldade de convergir matematicamente.<br>

Outro escalonador possível seria o `StandarScaler`, que redimensiona os dados para que tenha média zero e desvio padrão 1 (normalização). No caso do MNIST pode não ser o melhor caminho, pois as imagens do dataset tem todas um fundo preto (0), o que garante algum contraste para identificar o dígito. Estes pixels são predominantes. Com a normalização, estes pixels passariam a adotar valores muito baixos, mas ainda diferente de 0 e podem "borrar" as imagens e dificultar o treinamento.<br>

#### Fase 3: Implementação e Treinamento dos 3 Modelos

Serão utilizados 3 modelos de classificação neste projeto:
- Support Vector Machines (SVM): O classifcador desta familia é o SVC. Tem um funcionamento que busca traçar hiperplanos no espaço vetorial que permitem separar as classes, admitindo uma certa tolerância / margem em torno deste plano;
- Random Forest: conjunto de Árvores de Decisão (Decision Trees) que trabalham com a ideia de separar progressivamente o conjunto de dados utilziando o conceito de
entropia / pureza para entender como as amostras se assemelham. Por natureza é um algoritmo multiclasse;
- Multi-Layer Perceptron (MLP): Rede Neural Artificial tradicional. Aqui usaremos o modelo distribuído pelo Scikit Learn por questões de praticidade, compatibilidade e natureza relativamente simples do problema (no contexto de visã computacional).

As redes serão submetidas a uma procura por melhores parâmetros (Grid Search), a partir do qual serão selecionadas as melhores configurações para cada família de classificador.
Os 3 melhores modelos serão então treinados em todo o conjunto de treino para posterior avaliação

In [ ]:
''' Função que executa o Grid Search dos modelos.
    Recebe como parâmetro nome do modelo, modelo base, a grade de parâmetros para avaliar e conjunto de treino
    Retorna ao final o modelo com as melhores configurações
'''
def procura_melhores_parametros (nome_modelo, modelo, parametros, X_treino, y_treino):

    # Separa uma amostra de somente 9000 entradas para teste, para o Grid Search rodar mais rápido
    _, X_treino_gs, _, y_treino_gs = train_test_split(
    X_treino, y_treino, test_size=9000, stratify=y_treino, random_state=42
)
    # Configura o Grid Search com Validação Cruzada de 3 folds
    grid_search = GridSearchCV(
        estimator=modelo,
        param_grid=parametros,
        cv=3,
        scoring="accuracy",
        n_jobs=-1,  # Usa todos os núcleos do processador em paralelo
        verbose=3,
    )

    # Executa a busca
    print(f"\nIniciando a busca de parâmetros otimizados do modelo {nome_modelo}...")
    inicio = time.time()
    grid_search.fit(X_treino_gs, y_treino_gs)
    fim = time.time()

    print(f"\nGrid Search concluído em {(fim - inicio)/60:.2f} minutos")

    print(f"Melhor combinação encontrada: {grid_search.best_params_}")
    print(f"Melhor Acurácia de Validação Cruzada: {grid_search.best_score_:.4f}")

    # Retorna o melhor classificador desta família
    return grid_search.best_estimator_

In [ ]:
''' Função que treina o modeloem todo o conjunto de treino e faz as predições e métricas na base de teste
    Recebe como parâmetros o nome do classificador, o modelo já instanciado, e os conjuntos de treino e teste
    Retorna um dicionário com o nome do modelo e suas métricas, a matriz-confusão e a instância do classificacdor treinado
'''
def treina_modelo(nome_classificador, classificador, Xtreino, ytreino, Xteste, yteste):

    classificador_treinado = clone(classificador) # fundamental usar a função clone, para começar um modelo com pesos zerados.
    print(f'\nIniciando o treinamento final do modelo {nome_classificador} com toda a base de treino')
    inicio_treino = time.time()
    classificador_treinado.fit(Xtreino, ytreino)
    fim_treino = time.time()
    tempo_treino = fim_treino - inicio_treino
    print(f"Treinamento concluído com sucesso em {tempo_treino/60:.2f} minutos!")

    print('Calculando predições nos dados de treino')
    y_pred_train = classificador_treinado.predict(Xtreino)
    print('Calculando predições nos dados de teste')
    y_pred_test = classificador_treinado.predict(Xteste)

    acuracia_treino = classification_report(ytreino,y_pred_train, output_dict=True, zero_division=0)['accuracy']
    acuracia_teste = classification_report(yteste,y_pred_test, output_dict=True, zero_division=0)['accuracy']
    print (f'Acurácia treino: {acuracia_treino}')
    print (f'Acurácia teste: {acuracia_teste}')
    relatorio_metricas = classification_report(yteste,y_pred_test, output_dict=True, zero_division=0)['weighted avg']
    print (f'Métricas do modelo: \n{relatorio_metricas}')
    matriz_confusao = confusion_matrix(yteste, y_pred_test)

    return {
        'nome_modelo':nome_classificador,'acuracia':acuracia_teste,
        'precision':relatorio_metricas['precision'],'recall':relatorio_metricas['recall'],
        'f1-score': relatorio_metricas['f1-score'], 'tempo_treino':tempo_treino
        }, matriz_confusao, classificador_treinado

In [ ]:
''' Cria uma lista de dicionários, no qual cada item é uma família de modelos
    Cada um recebe o nome/tipo do modelo, um dicionário com parâmetros de gridsearch e o modelo base a ser instanciado
'''
modelos_testar = []
modelos_testar.append({
    'nome_modelo': 'SVM',
    'param_grid':{"kernel": ["linear", "rbf"],"C":[1,10]},
    # kernel rbf permite planos mais flexíveis. C trata do grau de regularização dos dados
    'modelo_base': SVC(cache_size=1000, random_state=42)
    # 2. Instancia o SVC alocando mais memória RAM para o cálculo (cache_size=1000)
     })

modelos_testar.append({
    'nome_modelo':'RandomForest',
    'param_grid': {'n_estimators':[50,150], # número de árvores
                    'max_depth':[10,20]}, # profundidade da árvore
    'modelo_base': RandomForestClassifier(n_jobs=-1, random_state=42)
    })


modelos_testar.append({
    'nome_modelo':'MLP',
    'param_grid': {
        "hidden_layer_sizes": [(64,), (100, 50)], # testa uma camada de 64 layers ocultos e duas camadas de 100 e 50 layers ocultos
        "learning_rate_init": [0.001, 0.01]}, # taxa de aprendizagem inicial. Ela vai sendo alterada dinamicamente durante o treino
    'modelo_base': MLPClassifier(
        activation="relu", # função de ativação dos neurônios. Matematicamente simples e atenua a oscilação
        solver="adam", # otimizador que atualiza os pesos e vieses dinamicamente e individualmente para cada neurônio
       early_stopping=True, # para o treinamento quando ele parar de melhorar. Economiza processamento e tempo
        random_state=42,
        verbose=True)
    })

In [ ]:
'''
    A lista de modelos vai ser percorrida e submetida à avaliação de parâmetros
    Os melhores de cada família farão parte de outra lista de modelos que serão treinados em toda a base
'''

melhores_modelos = []
for i in modelos_testar:
    nome_modelo = i['nome_modelo']
    param_grid = i['param_grid']
    modelo_base = i['modelo_base']

    melhor_modelo = procura_melhores_parametros(nome_modelo, modelo_base, param_grid, X_train_escalonado, y_train)
    melhores_modelos.append({'nome_modelo':nome_modelo, 'modelo': melhor_modelo})

In [ ]:
'''
Treinamento dos 3 melhores modelos de cada família em toda a base de treino
Serão geradas 3 listas contendo: as métricas de cada modelo, as matrizes de confusão, e as instâncias dos modelos treinados
'''
resultados_modelos = []
matrizes = []
modelos_treinados = []

for m in melhores_modelos:

    nome_classificador = m['nome_modelo']
    classificador = m['modelo']
    
    resultados_modelo, matriz_confusao, classificador_treinado = treina_modelo(
        nome_classificador, classificador, X_train_escalonado, y_train, X_test_escalonado, y_test)
    
    resultados_modelos.append(resultados_modelo)
    matrizes.append({'classificador':nome_classificador, 'matriz_confusao': matriz_confusao})
    modelos_treinados.append({'modelo': nome_classificador, 'modelo_treinado': classificador_treinado})

#### Fase 4: Avaliação Comparativa de Desempenho

Avaliação dos resultados do treinamento dos três modelos.
Será feita a plotagem das matrizes de confusão de cada um deles e registradas as métricas em uma tabela / dataframe

In [ ]:
'''
Função que plota a matriz de confusão no formato de mapa de calor.
Recebe uma matriz numérica e o nome do modelo
'''

def plota_matriz_confusao(matriz_confusao, nome_modelo):
 
    plt.figure(figsize=(10, 8))

    sns.heatmap(
        matriz_confusao, annot=True, fmt="d", cmap="Blues",
        linewidths=0.5, linecolor="silver", cbar=True, 
        xticklabels=list(range(10)), yticklabels=list(range(10)),
    )


    plt.title(
        f"Matriz de Confusão - Modelo {nome_modelo}\nProjeto DigIA",
        fontsize=14, pad=20, fontweight="bold",
        )
    plt.xlabel("Dígito Predito pelo Modelo", fontsize=12, labelpad=10)
    plt.ylabel("Dígito Real", fontsize=12, labelpad=10)

    plt.xticks(rotation=0)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
# chama a função de plotagem para todas as matrizes da lista
for m in matrizes:
    classificador_plotar = m['classificador']
    matriz_plotar = m['matriz_confusao']
    plota_matriz_confusao(matriz_plotar, classificador_plotar)

In [ ]:
# Exibe tabela com os resultados / métricas dos 3 modelos.
pd.DataFrame(resultados_modelos)

Os dígitos que causaram mas confusão (real x previsto) no modelo foram:
- SVM: 4x9 (17) / 7x9 (9) / 3x9 (9)
- RF: 4x9 (32) / 9x4 (19) / 8x9 (18)
- MLP: 4x9 (48) / 8x3 (22) / 3x2 (15)

Os modelos tiveram difculdades diversas, mas o maior desafio é a diferenciação entre os dígitos 4 e 9, que podem variar a grafia e se assemelharem.
Em todos os algoritmos as maiores confusões envolvem de alguma forma os dígitos 4, 9,  e 3 predominantemente

Quanto ao desempenho, como o MNIST apresenta classes balanceadas, o custo de erro entre dígitos é praticamente igual. Sendo assim, a acurácia é uma métrica válida para avaliar o desepenho global dos modelos. o F1-Score registrado é o ponderado (Poderia ser usado médio já que há balanceamento entre classes). Ou seja, ele avalia também se há um equilíbrio entre acertos em todas as classes (sem penalizações por falsos positivos e falsos negativos) e pode ser usado complementarmente à acurácia.

Considerando a acurácia em teste, o SVM foi o mais bem sucedido. No entanto, o tempo de treino é sensivelmente mais alto que os demais. Seria interessante investir mais tempo em testes de versões mais simplificadas do SVM e/ou versões mais sofisticadas do MLP. Por ora, escolheria o MLP.

#### Fase 5: Testes de generalização

##### Fase 5.1: Treinamento Restrito com Classes Ocultadas (Class Masking)

Serão ocultadas as classes 3 e 6 do treinamento. Os modelos serão retreinados e submetidos à predição de uma base que possui somente estas classes

In [ ]:
# ocultar 3 e 6
filtro_ood_treino = (y_train != 3) & (y_train != 6)
filtro_ood_teste = (y_test == 3) | (y_test == 6)

X_train_ood = X_train[filtro_ood_treino]
y_train_ood = y_train[filtro_ood_treino]
X_test_ood = X_test[filtro_ood_teste]
y_test_ood = y_test[filtro_ood_teste]

print(f"--- Separando conjunto de treino com classes 3 e 6 ocultas ---")
print(f"Amostras no treino original: {X_train.shape}")
print(f"Amostras no treino restrito (sem 3 e 6): {X_train_ood.shape}")
print(f"Classes presentes no novo treino: {np.unique(y_train_ood)}")

print('Escalonando conjuntos de treino e teste')
scaler_ood = MinMaxScaler()
X_train_ood_escalonado = scaler_ood.fit_transform(X_train_ood)
X_test_ood_escalonado = scaler_ood.transform(X_test_ood)



#####  5.2 Teste de Generalização Extrema (Inferência OOD)

In [ ]:
# Modelos retreinados
matrizes_ood = []
modelos_ood = []

for m in melhores_modelos:

    nome_classificador = m['nome_modelo']
    classificador = m['modelo']
    
    _, matriz_confusao, classificador_treinado = treina_modelo(
        nome_classificador, classificador, X_train_ood, y_train_ood, X_test_ood, y_test_ood)
    
    matrizes_ood.append({'classificador':nome_classificador, 'matriz_confusao': matriz_confusao})
    modelos_ood.append({'modelo': nome_classificador, 'modelo_treinado': classificador_treinado})

In [ ]:
# plota as matrizes de confusão novamente.

for m in matrizes_ood:
    classificador_plotar = m['classificador']
    matriz_plotar = m['matriz_confusao']
    plota_matriz_confusao(matriz_plotar, classificador_plotar)

Aqui podemos observar o que acontece quando o modelo é forçado a classificar algo que nunca viu.
Interessante observar que todos os modelos, em maior ou menor grau, tendem fazer as mesmas predições falsas:
- dígito 3: tende ao 8 e 5
- dígito 6: tende ao 4 e 5

#####  5.3 Inferência com Imagens Manuscritas Próprias

Para esta etapa foram coletados digitos manuscritos em uma folha quadriculada.
A folha foi digitalizada em um scanner de mesa com resolução de 300 dpi, já em escala de cinza (Grayscale).
Os digitos foram recortados manualmente no software GIMP de processamento de imagens 
e armaazenados em uma pasta específica do projeto.<br>

O script abaixo faz um processamento em massa destes arquivos produzindo a inversão de cores (fundo preto),
centralização do dígito pela massa dentro de uma grade de 28x28 pixels.<br>

In [ ]:
# Observação: Este função foi escrita quase integralmente por uma LLM

def processa_digito_manual(caminho_imagem):
    """Versão corrigida e calibrada para o padrão espacial exato do MNIST."""
    img = cv2.imread(caminho_imagem, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # Suaviza de leve para tirar imperfeições do papel
    img_blur = cv2.GaussianBlur(img, (3, 3), 0)

    # Threshold rígido com inversão (Garante fundo 100% preto de verdade)
    _, thresh = cv2.threshold(img_blur, 130, 255, cv2.THRESH_BINARY_INV)

     # Encontra a caixa envolvente
    contornos, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contornos) == 0:
        return np.zeros((28, 28), dtype=np.uint8)

    maior_contorno = max(contornos, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(maior_contorno)
    recorte_digito = thresh[y : y + h, x : x + w]

    # Redimensiona para proporção 20x20
    if w > h:
        novo_w = 20
        novo_h = int(h * (20 / w))
    else:
        novo_h = 20
        novo_w = int(w * (20 / h))

    novo_w, novo_h = max(1, novo_w), max(1, novo_h)
    digito_redimensionado = cv2.resize(
        recorte_digito, (novo_w, novo_h), interpolation=cv2.INTER_AREA
    )

    lona_mnist = np.zeros((28, 28), dtype=np.uint8)
    start_y = (28 - novo_h) // 2
    start_x = (28 - novo_w) // 2
    lona_mnist[start_y : start_y + novo_h, start_x : start_x + novo_w] = (
        digito_redimensionado
    )

    # Centralização por Centro de Massa
    cy, cx = ndimage.center_of_mass(lona_mnist)
    if not math.isnan(cy) and not math.isnan(cx):
        shift_y = int(round(14.0 - cy))
        shift_x = int(round(14.0 - cx))
        M = np.float32([[1, 0, shift_x], [0, 1, shift_y]])
        lona_mnist = cv2.warpAffine(lona_mnist, M, (28, 28))

     # Engrossa o traço diretamente na lona final de 28x28 para o modelo enxergar
    kernel = np.ones((2, 2), np.uint8)
    lona_mnist = cv2.dilate(lona_mnist, kernel, iterations=1)

    # Aplica um desfoque suave para recriar as bordas esfumaçadas do MNIST original
    lona_mnist = cv2.GaussianBlur(lona_mnist, (3, 3), 0)

    return lona_mnist


In [ ]:
# faz a varredura da pasta dos digitos e chama a função de processamento
PASTA_DIGITOS = "digitos-manuscritos"

extensoes = ("*.png", "*.jpg", "*.jpeg")
arquivos_encontrados = []
for ext in extensoes:
    arquivos_encontrados.extend(glob.glob(os.path.join(PASTA_DIGITOS, ext)))

digitos_processados = []
nomes_arquivos = []

# Processa em lote
print(f"Encontradas {len(arquivos_encontrados)} imagens na pasta para tratar.")
for caminho in sorted(arquivos_encontrados):
    digito_tratado = processa_digito_manual(caminho)
    if digito_tratado is not None:
        digitos_processados.append(digito_tratado)
        nomes_arquivos.append(os.path.basename(caminho))

print(f"{len(digitos_processados)} imagens prontas para o modelo!")

In [ ]:
# visualização em matriz dos digitos manuscritos
def visualiza_digitos_cropados(lista_digitos):
    """Plota uma grade com todos os dígitos detectados e tratados para validação visual."""
    total_digitos = len(lista_digitos)

    if total_digitos == 0:
        print("Nenhuma imagem na lista para plotar!")
        return

    colunas = 10
    linhas = math.ceil(total_digitos / colunas)

    plt.figure(figsize=(15, 1.5 * linhas))

    for idx, digito in enumerate(lista_digitos):
        plt.subplot(linhas, colunas, idx + 1)
        plt.imshow(digito, cmap="gray", vmin=0, vmax=255)
        plt.title(f"Idx: {idx}", fontsize=8)
        plt.axis("off")

    plt.suptitle(
        f"Verificação de dígitos manuscritos - {total_digitos} Dígitos Isolados",
        fontsize=14,
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

In [ ]:
visualiza_digitos_cropados(digitos_processados)

In [ ]:
# Processamento da lista de matrizes 28x28 em uma matriz de n dígitos por 784 (achatada)
X_fotos_proprias = np.vstack([mat.reshape(784) for mat in digitos_processados])
# Escalonamento
X_fotos_proprias_scaled = scaler.transform(X_fotos_proprias)

print(f"Dimensão final da matriz de fotos próprias: {X_fotos_proprias_scaled.shape}")


In [ ]:
# executa as predições usando os modelos já treinados (originais)
predicoes_finais = []
for m in melhores_modelos:
    predicao_final = m['modelo'].predict(X_fotos_proprias_scaled)
    predicoes_finais.append({'modelo_testado': m['nome_modelo'], 'predicao':predicao_final})

In [ ]:
def plota_resultados_finais(
        modelo_testado, predicoes, indices_aleatorios,
        quantidade_para_plotar, amostras):

    plt.figure(figsize=(15, 3))

    for i, idx in enumerate(indices_aleatorios):
        plt.subplot(1, quantidade_para_plotar, i + 1)
        imagem_28x28 = amostras[idx].reshape(28, 28)
        plt.imshow(imagem_28x28, cmap="gray", vmin=0.0, vmax=1.0)
        palpite = predicoes[idx]
        plt.title(f"\nPred: {palpite}", fontsize=12)
        plt.axis("off")

    plt.suptitle(
        f"Amostras Aleatórias de Teste Real Modelo {modelo_testado} - Projeto DigIA",
        fontsize=14,
        fontweight="bold",
        y=1.1,
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# Exibição de alguns resultados

total_amostras = X_fotos_proprias_scaled.shape[0]
quantidade_para_plotar = min(10, total_amostras)

if quantidade_para_plotar == 0:
    print("Nenhuma imagem processada disponível para exibição.")
else:
    indices_aleatorios = random.sample(range(total_amostras), quantidade_para_plotar)

    for p in predicoes_finais:
        plota_resultados_finais (p['modelo_testado'], p['predicao'], indices_aleatorios,
                                 quantidade_para_plotar, X_fotos_proprias_scaled)



Os resultados exibidos comparam as prediçõesem produção. Apenas para uma comparação rápida das taxas de acerto nas 10 amostras exibidas:
- SVM: 10/10
- RF: 7/10
- MLP: 9/10

Como mencionado anteriormente na avaliação do treinamento inicial dos modelos, a taxa de acerto real do MLP é alta em relação ao tempo de treinamento / processamento requerido.
Se utilizarmos um índice de desempenho baseado na acurácia em produção e o tempo de treinamento obteríamos:

- SVM: 0,0538
- MLP: 0,45

O RF obteria um índice melhor, mas com uma taxa de acerto insatisfatória.
Conclusão: considerando somente os dados e modelos testados neste projeto, o caminho que parece mais indicado é o de desenvolver um MLP simples ou uma rede neural com outro tipo de arquitetura, ainda enxuta e com treinamento mais veloz.